exp2

In [1]:
import json
from pathlib import Path
import random

def split_json_dataset(input_path, out_dir, seed=42):
    p = Path(input_path)
    with open(p, 'r', encoding='utf-8') as f:
        data = json.load(f)
    random.Random(seed).shuffle(data)
    n = len(data)
    n_train = int(n * 0.8)
    n_dev = int(n * 0.1)
    train = data[:n_train]
    dev = data[n_train:n_train + n_dev]
    test = data[n_train + n_dev:]
    out = Path(out_dir)
    out.mkdir(parents=True, exist_ok=True)
    with open(out / 'train.json', 'w', encoding='utf-8') as f:
        json.dump(train, f, ensure_ascii=False)
    with open(out / 'dev.json', 'w', encoding='utf-8') as f:
        json.dump(dev, f, ensure_ascii=False)
    with open(out / 'test.json', 'w', encoding='utf-8') as f:
        json.dump(test, f, ensure_ascii=False)
    return len(train), len(dev), len(test)

src = '/storage/dataset/dermatoscop/DermEmbeddingBenchmark/exp1-DermDiseaseDescription/Qwen-Qwen3-Omni-30B-A3B-Thinking_symptom_classification_results.json'
out_dir = '/storage/dataset/dermatoscop/DermEmbeddingBenchmark/exp1-DermDiseaseDescription'
split_sizes = split_json_dataset(src, out_dir)
print(split_sizes)


(1600, 200, 200)


exp3

In [ ]:
import pandas as pd
from pathlib import Path
import json

def export_csv_classification_json(csv_path, out_path):
    cols = ["exogenous", "hair_diseases", "hereditary", "inflammatory", "nail_diseases", "proliferations", "reaction_patterns_and_descriptive_terms"]
    df = pd.read_csv(csv_path)
    df[cols] = df[cols].apply(pd.to_numeric, errors='coerce').fillna(0).astype(int)
    records = []
    i=0
    for _, r in df.iterrows():
        # 构造与示例格式一致的 dict
        records.append({
            "id": int(i),
            "sentence1": r['caption'],      # 原文 caption 作为 sentence1                           # sentence2 留空（原数据无对应字段）
            "label": [int(r[c]) for c in cols]           # 7 维多标签列表
        })
        i+=1
    p = Path(out_path)
    random.shuffle(records)                       # 随机打乱
    total = len(records)
    train_end = int(total * 0.8)
    dev_end = int(total * 0.9)

    splits = {
        'train.json': records[:train_end],
        'dev.json': records[train_end:dev_end],
        'test.json': records[dev_end:]
    }

    for fname, data in splits.items():
        split_path = p / fname
        with open(split_path, 'w', encoding='utf-8') as f:
            for item in data:
                f.write(json.dumps(item, ensure_ascii=False) + '\n')

csv_path = Path('/storage/dataset/dermatoscop/DermEmbeddingBenchmark/exp3-Derm1MTopologyLevel1Classification/Derm1M_v2_pretrain-4cls-onehot.csv')
json_out = Path('/storage/dataset/dermatoscop/DermEmbeddingBenchmark/exp3-Derm1MTopologyLevel1Classification')
export_csv_classification_json(csv_path, json_out)


exp4

In [ ]:
##清洗文本数据
import re
import pandas as pd
from pathlib import Path


def clean_qa_response(text, max_len=3000):
    if not isinstance(text, str):
        return ''
    t = text.replace('\r', '')
    lines = t.split('\n')
    items = []
    flags = []
    for ln in lines:
        s = ln.strip()
        if not s:
            continue
        is_item = bool(re.match(r'^(\d+[.)]|[-\u2022])', s))
        s = re.sub(r'^(\d+[.)]|[-\u2022])\s*', '', s)
        items.append(s)
        flags.append(is_item)
    if not items:
        return ''
    out = items[0]
    for i in range(1, len(items)):
        sep_item = (flags[i] or flags[i-1])
        sep = '; ' if sep_item else ' '
        if sep_item:
            out = out.rstrip('.。;:')
        out += sep + items[i]
    out = re.sub(r'\s+', ' ', out).strip()
    out = re.sub(r'\s+([,;:])', r'\1', out)
    if out and not re.search(r'[.!?]$', out):
        out += '.'
    if len(out) > max_len:
        out = out[:max_len].rstrip()
    return out

def clean_exp4_csv(input_csv, output_csv):
    src = Path(input_csv)
    df = pd.read_csv(src)
    df['response'] = df['response'].apply(clean_qa_response)
    out_csv = Path(output_csv)
    df.to_csv(out_csv, index=False)
    return str(out_csv)

clean_exp4_csv('/storage/dataset/dermatoscop/DermEmbeddingBenchmark/exp4-DermQA/combined_data.csv', '/storage/dataset/dermatoscop/DermEmbeddingBenchmark/exp4-DermQA/combined_data_clean.csv')

'/storage/dataset/dermatoscop/DermEmbeddingBenchmark/exp4-DermQA/combined_data_clean.csv'

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

# 读取csv文件
df = pd.read_csv('/storage/dataset/dermatoscop/DermEmbeddingBenchmark/exp4-DermQA/combined_data.csv')  # 请将'your_file.csv'替换为实际文件名

# 先按8:2划分训练集和临时集
train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42, shuffle=True)

# 再将临时集按1:1划分验证集和测试集（即原数据的1:1）
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, shuffle=True)

# 保存划分后的数据（可选）
train_df.to_csv('/storage/dataset/dermatoscop/DermEmbeddingBenchmark/exp4-DermQA/train.csv', index=False)
val_df.to_csv('/storage/dataset/dermatoscop/DermEmbeddingBenchmark/exp4-DermQA/val.csv', index=False)
test_df.to_csv('/storage/dataset/dermatoscop/DermEmbeddingBenchmark/exp4-DermQA/test.csv', index=False)

处理原始的csv文件，随机抽取负样本